# Simple RNN for SMS Text Classification

**Problem statement:** Predict whether an SMS message is **ham** (legitimate) or
**spam** using the [SMS Spam Collection dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset).
Keep only text and label; tokenize with Keras `Tokenizer(num_words=5000)`;
pad/truncate every sequence to exactly 50 tokens; build
`Embedding(5000, 32) → SimpleRNN(32) → Dense(1, activation="sigmoid")`;
train for exactly **5 epochs** and report held-out test accuracy.

**Target:** label (`ham=0`, `spam=1`). **Input feature:** the SMS text only.
This dataset has two classes, so sigmoid and binary cross-entropy are used.
For a dataset with more than two classes, use `Dense(n_classes, activation="softmax")`
and sparse categorical cross-entropy with integer labels instead.

Run all cells in order. On Kaggle, attach `uciml/sms-spam-collection-dataset`.
Locally/Colab, upload `spam.csv` beside this notebook or let KaggleHub download it.
If dependencies are missing, run `%pip install tensorflow pandas scikit-learn kagglehub` first.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["OMP_NUM_THREADS"] = "1"

import json
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Small CPU thread pools also work on hosted notebook runtimes.
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)
SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__, "pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)

TensorFlow: 2.20.0
NumPy: 2.2.6 pandas: 3.0.6 scikit-learn: 1.9.1


## 1. Load only text and label

The CSV uses `v1` for the label and `v2` for the text. Drop the unrelated empty
columns by selecting these two columns explicitly. Remove missing/empty rows
and exact duplicate messages before splitting, so identical text cannot appear
in both training and testing. No stemming, TF-IDF, or extra input features are used.

In [2]:
candidates = [Path("spam.csv"), Path("data/spam.csv"), Path("data/sms_spam/spam.csv")]
if Path("/kaggle/input").exists():
    candidates = sorted(Path("/kaggle/input").rglob("spam.csv")) + candidates
csv_path = next((path for path in candidates if path.is_file()), None)
if csv_path is None:
    import kagglehub
    dataset_dir = Path(kagglehub.dataset_download("uciml/sms-spam-collection-dataset"))
    csv_path = next(dataset_dir.rglob("spam.csv"))

raw = pd.read_csv(csv_path, encoding="latin-1")
df = raw[["v2", "v1"]].rename(columns={"v2": "text", "v1": "label"}).copy()
raw_rows = len(df)
df = df.dropna(subset=["text", "label"])
df = df[df["text"].str.strip().ne("")].copy()
assert df.groupby("text")["label"].nunique().max() == 1, "Conflicting labels for identical text"
duplicates_removed = int(df.duplicated(subset=["text"]).sum())
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
label_mapping = {"ham": 0, "spam": 1}
assert set(df["label"]) == set(label_mapping)
df["label"] = df["label"].map(label_mapping).astype("int32")
assert df.columns.tolist() == ["text", "label"]
print(f"Raw rows: {raw_rows}; duplicate messages removed: {duplicates_removed}; retained: {len(df)}")
print("Label mapping:", label_mapping)
print(df["label"].value_counts().rename(index={0: "ham", 1: "spam"}))
df.head()

Raw rows: 5572; duplicate messages removed: 403; retained: 5169
Label mapping: {'ham': 0, 'spam': 1}
label
ham     4516
spam     653
Name: count, dtype: int64


,text,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


## 2. Split before learning the vocabulary

Use a reproducible, stratified 80% training / 20% test split. Stratification
preserves the ham/spam proportions; the test set is used only after five epochs.
The vocabulary is learned from training text only.

In [3]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.20, random_state=SEED, stratify=df["label"]
)
assert set(X_train_text).isdisjoint(set(X_test_text))
print(f"Training messages: {len(X_train_text)}; test messages: {len(X_test_text)}")

tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)
train_sequences = tokenizer.texts_to_sequences(X_train_text)
test_sequences = tokenizer.texts_to_sequences(X_test_text)

Training messages: 4135; test messages: 1034


## 3. Make every input exactly 50 tokens

The same `maxlen=50` handles every row: shorter sequences get zeros at the
beginning and longer sequences keep their last 50 tokens. Keras reserves index
0 for padding; `num_words=5000` retains word indices below 5000, including the
out-of-vocabulary token at index 1, so all indices fit the 5,000-row embedding.

In [4]:
X_train = pad_sequences(train_sequences, maxlen=50, padding="pre", truncating="pre")
X_test = pad_sequences(test_sequences, maxlen=50, padding="pre", truncating="pre")
y_train = y_train.to_numpy(dtype="int32")
y_test = y_test.to_numpy(dtype="int32")
assert X_train.shape == (len(y_train), 50)
assert X_test.shape == (len(y_test), 50)
assert 0 <= X_train.min() and X_train.max() < 5000
assert 0 <= X_test.min() and X_test.max() < 5000
print("Training shape:", X_train.shape, "Test shape:", X_test.shape)
print("First test sequence length before padding:", len(test_sequences[0]))
print("First test input after padding/truncation:", X_test[0])

Training shape: (4135, 50) Test shape: (1034, 50)
First test sequence length before padding: 31
First test input after padding/truncation: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0   59  204   13   71    2   51    3  234   23
    8  297    4    5  123   66  748   16  553  335    8 1671    4   22
   13  190  280  177    2   71    4  455]


## 4. Build the exact requested model

Each token becomes a 32-value embedding. The 32-unit SimpleRNN reads the
50-token sequence and passes its final state to a single sigmoid output,
which estimates the probability of spam. There are no additional model layers.

In [5]:
model = Sequential([
    Embedding(5000, 32),
    SimpleRNN(32),
    Dense(1, activation="sigmoid")
])
model.build(input_shape=(None, 50))
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()
assert [type(layer).__name__ for layer in model.layers] == ["Embedding", "SimpleRNN", "Dense"]
assert model.count_params() == 162113

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 32)         │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 162,113 (633.25 KB)

 Trainable params: 162,113 (633.25 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train for exactly five epochs

Batch size is 64. The TensorFlow datasets below only batch/shuffle the inputs;
their small thread pools keep CPU execution reliable. No early stopping or test
set validation is used, and there is no tuning against test results.

In [6]:
options = tf.data.Options()
options.threading.private_threadpool_size = 1
options.threading.max_intra_op_parallelism = 1
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
train_ds = train_ds.batch(64).with_options(options)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(64).with_options(options)

history = model.fit(train_ds, epochs=5, verbose=2)
assert len(history.epoch) == 5

Epoch 1/5


65/65 - 1s - 14ms/step - accuracy: 0.8984 - loss: 0.3296


Epoch 2/5


65/65 - 0s - 3ms/step - accuracy: 0.9748 - loss: 0.0976


Epoch 3/5


65/65 - 0s - 3ms/step - accuracy: 0.9879 - loss: 0.0534


Epoch 4/5


65/65 - 0s - 3ms/step - accuracy: 0.9920 - loss: 0.0349


Epoch 5/5


65/65 - 0s - 3ms/step - accuracy: 0.9896 - loss: 0.0335


## 6. Report held-out test accuracy

Classify probabilities of at least 0.5 as spam. Since ham is more frequent,
also show a majority-class baseline and per-class precision/recall/F1 to give
the required accuracy some context. Results can vary slightly across platforms.

In [7]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
probabilities = model.predict(test_ds, verbose=0).reshape(-1)
y_pred = (probabilities >= 0.5).astype("int32")
assert np.isclose(test_accuracy, accuracy_score(y_test, y_pred), atol=1e-6)
majority_label = int(np.bincount(y_train).argmax())
baseline_accuracy = float(np.mean(y_test == majority_label))
print(f"TEST ACCURACY: {test_accuracy:.4f} ({test_accuracy:.2%})")
print(f"Majority-class baseline accuracy: {baseline_accuracy:.4f} ({baseline_accuracy:.2%})")
print("Confusion matrix (rows=actual, columns=predicted; order=ham, spam):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["ham", "spam"], digits=4, zero_division=0))

output_dir = Path("simple_rnn_results")
output_dir.mkdir(exist_ok=True)
summary = {
    "dataset": "uciml/sms-spam-collection-dataset",
    "raw_rows": raw_rows, "duplicates_removed": duplicates_removed, "retained_rows": len(df),
    "train_rows": len(X_train), "test_rows": len(X_test), "seed": SEED,
    "num_words": 5000, "maxlen": 50, "epochs": len(history.epoch), "batch_size": 64,
    "architecture": "Embedding(5000,32) -> SimpleRNN(32) -> Dense(1,sigmoid)",
    "model_parameters": model.count_params(), "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss), "majority_baseline_accuracy": baseline_accuracy,
    "confusion_matrix": confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist(),
    "tensorflow_version": tf.__version__,
}
(output_dir / "results_summary.json").write_text(json.dumps(summary, indent=2))
pd.DataFrame(history.history, index=range(1, 6)).rename_axis("epoch").to_csv(output_dir / "training_history.csv")

TEST ACCURACY: 0.9768 (97.68%)
Majority-class baseline accuracy: 0.8733 (87.33%)
Confusion matrix (rows=actual, columns=predicted; order=ham, spam):
[[903   0]
 [ 24 107]]
              precision    recall  f1-score   support

         ham     0.9741    1.0000    0.9869       903
        spam     1.0000    0.8168    0.8992       131

    accuracy                         0.9768      1034
   macro avg     0.9871    0.9084    0.9430      1034
weighted avg     0.9774    0.9768    0.9758      1034



## Why plain RNNs struggle with long sequences

During backpropagation through time, gradients are repeatedly multiplied by
recurrent weights and activation derivatives, which can make them shrink toward
zero over many time steps (the vanishing gradient problem). As a result, words
far back in a long sequence receive weak learning signals, making long-range
dependencies difficult for a plain RNN to learn.

## Assignment checklist

- Only `text` and `label` columns retained.
- Keras `Tokenizer(num_words=5000)` fitted on training text only.
- Both partitions use `pad_sequences(..., maxlen=50)` with shape assertions.
- Exact `Embedding(5000, 32) → SimpleRNN(32) → Dense(1, sigmoid)` model.
- Exactly five epochs; held-out test accuracy printed above.
- Two-sentence explanation of vanishing gradients.

References: [dataset](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset),
[Keras Tokenizer](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer),
[pad_sequences](https://www.tensorflow.org/api_docs/python/tf/keras/utils/pad_sequences),
[SimpleRNN](https://www.tensorflow.org/api_docs/python/tf/keras/layers/SimpleRNN).
`Tokenizer` is a legacy API, used here to match the assignment explicitly.